# Linear Regression – Programming Assignment, Part 2

## 1. Introduction

This notebook predicts daily bicycle rentals using polynomial regression. It first checks whether a polynomial can interpolate the training data, then compares ordinary least squares and Ridge and Lasso regularization. The five approaches use the same design matrices and the training and test split from Part 1.

## 2. Dataset Description

The UCI Bike Sharing dataset records rentals from Capital Bikeshare in Washington, D.C., during 2011 and 2012. Capital Bikeshare's operating software logged individual trips, including their start and end times, stations, and rider membership type. Hadi Fanaee-T and João Gama aggregated those public trip records into hourly and daily counts and added weather and calendar information. They matched the rental time series to historical weather reports from Freemeteo and used the District of Columbia's official holiday schedule. The processed dataset supports rental demand prediction and was used in their study of event detection in rental activity. [Original study, section 3.1](https://doi.org/10.1007/s13748-013-0040-3)

The daily file, **day.csv**, contains **731 rows and 16 columns**, with one row per day. Part 2 uses **temp** and **hum** as inputs. The target remains **cnt**. [Fanaee-T, H. (2013). Bike Sharing dataset. UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/day.csv")
print(f"Dataset dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
first_days = df.head()
print(first_days.iloc[:, :8].to_string())
print()
print(first_days.iloc[:, 8:].to_string())

Dataset dimensions: 731 rows, 16 columns
   instant      dteday  season  yr  mnth  holiday  weekday  workingday
0        1  2011-01-01       1   0     1        0        6           0
1        2  2011-01-02       1   0     1        0        0           0
2        3  2011-01-03       1   0     1        0        1           1
3        4  2011-01-04       1   0     1        0        2           1
4        5  2011-01-05       1   0     1        0        3           1

   weathersit      temp     atemp       hum  windspeed  casual  registered   cnt
0           2  0.344167  0.363625  0.805833   0.160446     331         654   985
1           2  0.363478  0.353739  0.696087   0.248539     131         670   801
2           1  0.196364  0.189405  0.437273   0.248309     120        1229  1349
3           1  0.200000  0.212122  0.590435   0.160296     108        1454  1562
4           1  0.226957  0.229270  0.436957   0.186900      82        1518  1600


| Column | Meaning |
| --- | --- |
| instant | Sequential record index. |
| dteday | Calendar date in YYYY-MM-DD format. |
| season | In the daily CSV, 1 = winter, 2 = spring, 3 = summer, 4 = fall. |
| yr | 0 = 2011, 1 = 2012. |
| mnth | Month of the year, 1 = January through 12 = December. |
| holiday | 1 for an official holiday, 0 otherwise. |
| weekday | Day of the week, 0 = Sunday through 6 = Saturday, based on the dates in the file. |
| workingday | 1 if the day is neither a weekend nor a holiday, 0 otherwise. |
| weathersit | Weather category. 1 = clear or partly cloudy, 2 = mist with varying cloud cover, 3 = light snow or light rain, possibly with a thunderstorm, 4 = severe weather such as heavy rain with ice pellets or snow with fog. |
| temp | Normalized daily mean air temperature (input). The archive documentation describes the daily scale as °C divided by 41. |
| atemp | Normalized daily mean apparent (feels-like) temperature. The archive documentation describes the daily scale as °C divided by 50. |
| hum | Daily mean relative humidity, expressed as a percentage divided by 100 (added input). |
| windspeed | Daily mean wind speed divided by 67. |
| casual | Daily number of rentals by casual users. |
| registered | Daily number of rentals by registered users. |
| cnt | Total rentals per day, equal to casual plus registered rentals (target). |

[UCI variable documentation](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset) and [original dataset archive with Readme.txt](https://archive.ics.uci.edu/static/public/275/bike%2Bsharing%2Bdataset.zip).

## 3. Enriching the Hypothesis Space

Part 1 used powers of normalized temperature. The split below reproduces the Part 1 shuffle and rounding. For any input value shared by several training days, a temperature-only model must return the same prediction on those days. The common prediction that minimizes their squared error is their mean rental count. Using the mean at every distinct temperature therefore gives a lower bound on training RMSE for any temperature-only function.

The number of distinct training temperatures also bounds the row rank. The degree checks below compare raw powers and Chebyshev polynomials. Chebyshev polynomials span the same univariate polynomial space and are evaluated by the recurrence $T_0(u)=1$, $T_1(u)=u$, and $T_j(u)=2uT_{j-1}(u)-T_{j-2}(u)$. For these checks and the enriched basis, each input is mapped using the midpoint and 110% of the half-range of its training values. The extra range keeps the training endpoints inside $[-1,1]$. The condition reported below is the largest singular value divided by the smallest one retained by the least-squares solver. The numerical rank shows how many were retained.

In [2]:
temperatures = df["temp"].to_numpy()
rentals = df["cnt"].to_numpy()

indices = np.arange(len(df))
random_generator = np.random.default_rng(42)
random_generator.shuffle(indices)
split_index = int(0.8 * len(indices))

train_indices = indices[:split_index]
test_indices = indices[split_index:]
x_train, y_train = temperatures[train_indices], rentals[train_indices]
x_test, y_test = temperatures[test_indices], rentals[test_indices]

print(f"Training samples: {len(train_indices)}")
print(f"Test samples: {len(test_indices)}")

Training samples: 584
Test samples: 147


In [3]:
def basis_function(x, j):
    return x ** j


def power_design_matrix(x, degree):
    matrix = np.empty((len(x), degree + 1))
    for j in range(degree + 1):
        matrix[:, j] = basis_function(x, j)
    return matrix


def sse(y, y_pred):
    return np.sum((y - y_pred) ** 2)


def rmse(y, y_pred):
    return np.sqrt(sse(y, y_pred) / len(y))


def predict(X, weights):
    return X @ weights


def chebyshev_values(x, degree):
    values = np.empty((len(x), degree + 1))
    values[:, 0] = 1
    if degree >= 1:
        values[:, 1] = x
    for j in range(2, degree + 1):
        values[:, j] = 2 * x * values[:, j - 1] - values[:, j - 2]
    return values

In [4]:
temperature_groups = df.iloc[train_indices].groupby("temp")["cnt"]
duplicate_groups = (temperature_groups.nunique() > 1).sum()
temperature_floor = rmse(y_train, temperature_groups.transform("mean").to_numpy())
print(f"Distinct training temperatures: {temperature_groups.ngroups}")
print(f"Temperatures with conflicting rental counts: {duplicate_groups}")
print(f"Temperature-only lower bound on training RMSE: {temperature_floor:.2f}")

scaled_temp = (2 * x_train - (x_train.max() + x_train.min())) / (
    1.10 * (x_train.max() - x_train.min())
)
temperature_checks = []
for degree in [15, 50, 100, len(y_train) - 1]:
    for name, basis in [("Powers", power_design_matrix), ("Chebyshev", chebyshev_values)]:
        matrix = basis(x_train if name == "Powers" else scaled_temp, degree)
        weights, _, rank, singular_values = np.linalg.lstsq(matrix, y_train, rcond=None)
        temperature_checks.append(
            (name, degree, matrix.shape[1], rank, singular_values[0] / singular_values[rank - 1],
             rmse(y_train, predict(matrix, weights)))
        )

diagnostic_columns = ["Basis", "Degree", "Columns", "Rank", "Condition", "Training RMSE"]
temperature_table = pd.DataFrame(temperature_checks, columns=diagnostic_columns)
print(temperature_table.to_string(
    index=False, formatters={"Condition": "{:.2e}".format, "Training RMSE": "{:,.2f}".format}
))

Distinct training temperatures: 433
Temperatures with conflicting rental counts: 118
Temperature-only lower bound on training RMSE: 716.95
    Basis  Degree  Columns  Rank Condition Training RMSE
   Powers      15       16    16  5.58e+12      1,411.08
Chebyshev      15       16    16  1.37e+03      1,411.08
   Powers      50       51    21  2.09e+12      1,408.15
Chebyshev      50       51    50  2.02e+11      1,388.16
   Powers     100      101    23  4.56e+12      1,407.31
Chebyshev     100      101    86  8.37e+11      1,337.63
   Powers     583      584    23  4.16e+12      1,407.28
Chebyshev     583      584   331  7.42e+12        993.72


The temperature-only error bound rules out interpolation. The power matrices also lose numerical rank as the degree increases. The Chebyshev representation improves conditioning at degree 15, but increasing the degree still leads to rank loss and leaves substantial training error.

Add one predictor, normalized relative humidity. It describes weather that may be associated with cycling demand and distinguishes training days with the same temperature. The two original inputs are scaled as above. For scaled temperature $u$ and scaled humidity $v$, the enriched basis consists of products $T_a(u)T_b(v)$ with $a+b\le d$. For each nonconstant product, subtract its training mean and use that same mean on the test set. Centering keeps the intercept separate and preserves the polynomial space. The first column is a constant. The record index, date, and components of the target are excluded.

To find a reproducible interpolating fit, start at degree 40, near the low end for a two-input polynomial with at least 584 terms. Increase the degree in steps of ten and accept the first candidate with full row rank, training RMSE below $10^{-3}$ rentals per day, and a condition number at most $10^7$. This condition check guards against accepting a fit that depends on nearly vanishing singular values. All criteria use the training data.

In [5]:
features = df[["temp", "hum"]].to_numpy()
train_features = features[train_indices]
print(f"Distinct training (temperature, humidity) pairs: {len(np.unique(train_features, axis=0))}")
feature_low = train_features.min(axis=0)
feature_high = train_features.max(axis=0)
feature_center = (feature_high + feature_low) / 2
feature_half_width = 1.10 * (feature_high - feature_low) / 2
scaled_train = (train_features - feature_center) / feature_half_width


def design_matrix(x, degree, orders):
    temperature_basis = chebyshev_values(x[:, 0], degree)
    humidity_basis = chebyshev_values(x[:, 1], degree)
    matrix = np.column_stack([temperature_basis[:, a] * humidity_basis[:, b] for a, b in orders])
    return np.ascontiguousarray(matrix)


enriched_checks = []
for degree in range(40, 81, 10):
    orders = [(a, d - a) for d in range(degree + 1) for a in range(d, -1, -1)]
    X_train = design_matrix(scaled_train, degree, orders)
    basis_means = X_train[:, 1:].mean(axis=0)
    X_train[:, 1:] -= basis_means
    ols_weights, _, rank, singular_values = np.linalg.lstsq(X_train, y_train, rcond=None)
    train_rmse = rmse(y_train, predict(X_train, ols_weights))
    condition = singular_values[0] / singular_values[rank - 1]
    enriched_checks.append(("Temperature and humidity", degree, len(orders), rank, condition, train_rmse))
    if rank == len(y_train) and condition <= 1e7 and train_rmse < 1e-3:
        break
else:
    raise RuntimeError("The candidate bases did not meet the training interpolation criteria.")

print("Temperature and humidity basis")
enriched_table = pd.DataFrame(enriched_checks, columns=diagnostic_columns).drop(columns="Basis")
print(enriched_table.to_string(index=False, float_format="{:.3e}".format))

Distinct training (temperature, humidity) pairs: 584
Temperature and humidity basis
 Degree  Columns  Rank  Condition  Training RMSE
     40      861   537  4.601e+12      3.513e+02
     50     1326   584  1.336e+12      5.660e-03
     60     1891   584  1.347e+09      4.130e-06
     70     2556   584  7.006e+06      3.576e-08


In [6]:
scaled_test = (features[test_indices] - feature_center) / feature_half_width
X_test = design_matrix(scaled_test, degree, orders)
X_test[:, 1:] -= basis_means

print(f"Selected degree {degree} with {X_train.shape[1]} columns including the intercept")
print(f"Unregularized training RMSE: {train_rmse:.3e} rentals per day")
print(f"Unregularized test RMSE: {rmse(y_test, predict(X_test, ols_weights)):,.2f} rentals per day")

Selected degree 70 with 2556 columns including the intercept
Unregularized training RMSE: 3.576e-08 rentals per day
Unregularized test RMSE: 32,364,181.13 rentals per day


## 4. Mathematical Background and Derivations

**Prediction and MSE.** Let $X\in\mathbb{R}^{N\times p}$ have one training sample per row, with $\mathbf{x}_i^\top$ denoting row $i$. Its first column is one. The model is $\hat{\mathbf{y}}=X\mathbf{w}$, linear in the weights $\mathbf{w}$. We use

$$
M(\mathbf{w})=\frac{1}{N}\sum_{i=1}^{N}(\mathbf{x}_i^\top\mathbf{w}-y_i)^2,
\qquad \mathrm{RMSE}=\sqrt{M(\mathbf{w})}.
$$

Differentiating each squared residual with respect to $w_j$ gives
$\frac{\partial M}{\partial w_j}=\frac{2}{N}\sum_{i=1}^{N}(\mathbf{x}_i^\top\mathbf{w}-y_i)X_{ij}$.
Thus $\nabla M(\mathbf{w})=\frac{2}{N}X^\top(X\mathbf{w}-\mathbf{y})$.
For a uniformly sampled training row $i$, the single-sample gradient is
$\mathbf{g}_i(\mathbf{w})=2(\mathbf{x}_i^\top\mathbf{w}-y_i)\mathbf{x}_i$.
Its expectation over $i$ is $\nabla M(\mathbf{w})$.

**Ridge (CF, GD-R, SGD-R).** Let $D=\operatorname{diag}(0,1,\dots,1)$ so the intercept is excluded. For $J_R=M+\lambda\mathbf{w}^\top D\mathbf{w}$, the penalty gradient is $2\lambda D\mathbf{w}$. Setting the full gradient to zero and multiplying by $N/2$ yields

$$
\frac{2}{N}X^\top(X\mathbf{w}-\mathbf{y})+2\lambda D\mathbf{w}=0
\quad\Longrightarrow\quad
(X^\top X+N\lambda D)\mathbf{w}=X^\top\mathbf{y}.
$$

For $\lambda>0$ the system is nonsingular. A vector with a nonzero penalized component has positive penalty curvature, and the constant column gives positive curvature for a vector with only an intercept. At $\lambda=0$ the least-squares solver returns a minimum-norm solution. With learning rate $\eta$, GD-R subtracts $\eta(\nabla M+2\lambda D\mathbf{w})$ per update. SGD-R instead subtracts $\eta(\mathbf{g}_i+2\lambda D\mathbf{w})$, keeping the full penalty gradient at every update.

**Lasso (proximal GD-L and proximal SGD-L).** The objective is
$J_L=M+\lambda\sum_{j=1}^{p-1}|w_j|$. The one-sided derivatives of $|t|$ at zero are $-1$ and $1$, so it has no derivative there. Its subgradient is $\{-1\}$ for $t<0$, $[-1,1]$ for $t=0$, and $\{1\}$ for $t>0$.

GD-L first takes the full MSE gradient step
$\mathbf{z}=\mathbf{w}-\eta\frac{2}{N}X^\top(X\mathbf{w}-\mathbf{y})$.
SGD-L takes a single-sample step
$\mathbf{z}=\mathbf{w}-2\eta(\mathbf{x}_i^\top\mathbf{w}-y_i)\mathbf{x}_i$.
Both then set $w_0^+=z_0$. For each penalized coordinate, the proximal step minimizes $\frac{1}{2}(t-z_j)^2+\eta\lambda|t|$. The subgradient condition $0\in t-z_j+\eta\lambda\,\partial|t|$ gives three cases.

- For $t>0$, $t=z_j-\eta\lambda$, which requires $z_j>\eta\lambda$.
- For $t<0$, $t=z_j+\eta\lambda$, which requires $z_j<-\eta\lambda$.
- For $t=0$, $0\in-z_j+\eta\lambda[-1,1]$, which holds when $|z_j|\le\eta\lambda$.

Together these give $w_j^+=\operatorname{sign}(z_j)\max(|z_j|-\eta\lambda,0)$ for $j\ge 1$. The same threshold is applied after every full-batch or single-sample MSE step. Exact zeros arise from the update itself.

## 5. Implementation of the Five Learning Approaches

The error and prediction functions are reused from Part 1. Ridge applies its penalty gradient to the non-intercept coefficients. The two Lasso implementations share the same soft-thresholding operation.

In [7]:
def train_cf(X, y, lam):
    if lam == 0:
        weights, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        return weights
    system = X.T @ X
    penalized = np.arange(1, X.shape[1])
    system[penalized, penalized] += len(y) * lam
    return np.linalg.solve(system, X.T @ y)


def mse_gradient(X, y, weights):
    return (2 / len(y)) * (X.T @ (predict(X, weights) - y))


def soft_threshold(values, threshold):
    return np.sign(values) * np.maximum(np.abs(values) - threshold, 0)


def train_gd_ridge(X, y, lam, learning_rate, passes):
    weights = np.zeros(X.shape[1])
    for _ in range(passes):
        gradient = mse_gradient(X, y, weights)
        gradient[1:] += 2 * lam * weights[1:]
        weights -= learning_rate * gradient
    return weights


def train_gd_lasso(X, y, lam, learning_rate, passes):
    weights = np.zeros(X.shape[1])
    for _ in range(passes):
        weights -= learning_rate * mse_gradient(X, y, weights)
        weights[1:] = soft_threshold(weights[1:], learning_rate * lam)
    return weights


def train_sgd_ridge(X, y, lam, learning_rates, sample_indices):
    weights = np.zeros(X.shape[1])
    for learning_rate, indices_in_pass in zip(learning_rates, sample_indices):
        shrinkage = 1 - 2 * learning_rate * lam
        for i in indices_in_pass:
            row = X[i]
            residual = row @ weights - y[i]
            weights[1:] *= shrinkage
            weights -= (2 * learning_rate * residual) * row
    return weights


def train_sgd_lasso(X, y, lam, learning_rates, sample_indices):
    weights = np.zeros(X.shape[1])
    for learning_rate, indices_in_pass in zip(learning_rates, sample_indices):
        threshold = learning_rate * lam
        for i in indices_in_pass:
            row = X[i]
            residual = row @ weights - y[i]
            weights -= (2 * learning_rate * residual) * row
            if threshold > 0:
                weights[1:] = soft_threshold(weights[1:], threshold)
    return weights

## 6. Training Procedure and Experimental Settings

The iterative methods start with all weights at zero and stop at the same sample-gradient budget. One GD update evaluates $N$ sample gradients. An SGD pass is $N$ single-sample updates, with each index drawn independently and uniformly with replacement. All SGD runs use the same sequence from seed 123. CF solves the system directly and has no gradient budget.

Let $L=2\sigma_{\max}^2(X)/N$ and $B=2\max_i\|\mathbf{x}_i\|_2^2$. These bound the full-batch and individual-sample MSE gradient curvatures, respectively. The GD learning rate is $1.75/(L+2\lambda)$ for Ridge and $1.75/L$ for Lasso, below twice the inverse curvature bound. The initial SGD rates are $1.5/(B+2\lambda)$ for Ridge and $1.5/B$ for Lasso, and are divided by $(1+k/20)^{0.65}$ at pass $k$ starting from $k=0$. The decrease reduces the effect of single-row gradient noise near the end.

The shared grid includes zero, the four positive values from the handout, and two larger values for Lasso. With an intercept-only fit, the Lasso threshold that sets all other coefficients to zero is $\lambda_{\max}=\max_{j\ge 1}|\nabla_j M|$ at $w_0=\bar{y}$. This training-only scale motivates the upper end of the grid. Ridge and Lasso have different penalties, so the same numeric $\lambda$ can exert a different strength.

In [8]:
lambdas = [0, 0.01, 0.1, 1, 10, 100, 1000]
passes = 1200
N, p = X_train.shape
sample_indices = np.random.default_rng(123).integers(N, size=(passes, N))
batch_curvature = 2 * singular_values[0] ** 2 / N
sample_curvature = 2 * np.max(np.sum(X_train ** 2, axis=1))
decay = (1 + np.arange(passes) / 20) ** 0.65
lasso_cutoff = np.max(np.abs((2 / N) * (X_train[:, 1:].T @ (y_train - y_train.mean()))))

print(f"Sample gradient evaluations per iterative fit: {passes * N:,}")
print(f"GD updates: {passes:,}   SGD updates: {passes * N:,}")
print(f"Training-only intercept-only Lasso cutoff: {lasso_cutoff:,.2f}")

models = {}
with np.errstate(over="raise", invalid="raise"):
    for lam in lambdas:
        models["CF", lam] = train_cf(X_train, y_train, lam)
        models["GD-R", lam] = train_gd_ridge(
            X_train, y_train, lam, 1.75 / (batch_curvature + 2 * lam), passes
        )
        models["GD-L", lam] = train_gd_lasso(
            X_train, y_train, lam, 1.75 / batch_curvature, passes
        )
        models["SGD-R", lam] = train_sgd_ridge(
            X_train, y_train, lam, 1.5 / ((sample_curvature + 2 * lam) * decay), sample_indices
        )
        models["SGD-L", lam] = train_sgd_lasso(
            X_train, y_train, lam, 1.5 / (sample_curvature * decay), sample_indices
        )

Sample gradient evaluations per iterative fit: 700,800
GD updates: 1,200   SGD updates: 700,800
Training-only intercept-only Lasso cutoff: 1,771.18


## 7. Experimental Results

Table 1 gives test RMSE in rentals per day. Table 3 gives training RMSE in the same units. CF at $\lambda=0$ is displayed in scientific notation to show the interpolation error.

In [9]:
methods = ["CF", "GD-R", "GD-L", "SGD-R", "SGD-L"]
lasso_methods = ["GD-L", "SGD-L"]
train_errors = pd.DataFrame(index=methods, columns=lambdas, dtype=float)
test_errors = train_errors.copy()
objectives = train_errors.copy()

for (method, lam), weights in models.items():
    train_predictions = predict(X_train, weights)
    test_predictions = predict(X_test, weights)
    assert np.isfinite(weights).all()
    assert np.isfinite(train_predictions).all() and np.isfinite(test_predictions).all()
    train_errors.loc[method, lam] = rmse(y_train, train_predictions)
    test_errors.loc[method, lam] = rmse(y_test, test_predictions)
    penalty = np.abs(weights[1:]) if method in lasso_methods else weights[1:] ** 2
    objectives.loc[method, lam] = sse(y_train, train_predictions) / N + lam * np.sum(penalty)

assert np.isfinite(objectives.to_numpy()).all()
assert (objectives.to_numpy() < sse(y_train, np.zeros(N)) / N).all()

print("Table 1. Test RMSE (rentals per day)")
test_table = test_errors.rename(columns=lambda lam: f"{lam:g}").rename_axis("Method")
print(test_table.to_string(float_format="{:,.2f}".format))

Table 1. Test RMSE (rentals per day)
                   0     0.01      0.1        1       10      100     1000
Method                                                                    
CF     32,364,181.13 2,411.38 1,708.25 1,644.46 1,886.65 2,005.83 2,022.67
GD-R        3,690.19 2,404.72 1,708.25 1,644.46 1,886.65 2,005.83 2,133.36
GD-L        3,690.19 3,674.48 3,535.30 2,655.83 1,507.08 1,238.01 1,688.94
SGD-R       3,496.88 2,399.29 1,709.44 1,639.95 1,884.46 2,007.74 2,023.46
SGD-L       3,496.88 3,483.60 3,366.96 2,593.26 1,502.12 1,235.90 1,684.96


In [10]:
print("Table 3. Training RMSE (rentals per day)")
training_table = train_errors.rename(columns=lambda lam: f"{lam:g}").rename_axis("Method")
print(training_table.to_string(
    float_format=lambda value: f"{value:,.2f}" if value >= 0.01 else f"{value:.2e}"
))

Table 3. Training RMSE (rentals per day)
              0   0.01    0.1        1       10      100     1000
Method                                                           
CF     3.58e-08 649.42 760.91 1,055.57 1,642.92 1,877.33 1,909.44
GD-R     602.41 650.00 760.91 1,055.57 1,642.92 1,877.33 1,988.55
GD-L     602.41 602.64 604.90   632.78   770.86 1,207.88 1,640.40
SGD-R    624.36 668.12 779.05 1,071.09 1,653.64 1,883.69 1,911.35
SGD-L    624.36 624.57 626.64   652.31   783.30 1,200.36 1,637.71


**Convergence check.** For Ridge, compare training objectives with CF at the same $\lambda$. For Lasso, a dual bound gives a reference for both methods without another fit. Let $X_{\mathrm{pen}}$ contain the penalized columns of $X$. For any $\mathbf{q}$ with $\mathbf{1}^\top\mathbf{q}=0$ and $\|X_{\mathrm{pen}}^\top\mathbf{q}\|_\infty\le\lambda$, completing the square gives

$$
M(\mathbf{w})\ge \mathbf{q}^\top(\mathbf{y}-X\mathbf{w})-\frac{N}{4}\|\mathbf{q}\|_2^2.
$$

The constraints ensure that the penalty cancels or exceeds the linear term in the weights. Thus $\mathbf{q}^\top\mathbf{y}-\frac{N}{4}\|\mathbf{q}\|_2^2$ is a lower bound on $J_L$. For each fit, take twice the residual divided by $N$, subtract its mean, and scale it down when needed to meet the constraints. Use the larger of the two bounds at each $\lambda$.

In [11]:
ridge_methods = ["GD-R", "SGD-R"]
positive_lambdas = lambdas[1:]
ridge_reference = objectives.loc["CF", positive_lambdas]
ridge_excess = 100 * (objectives.loc[ridge_methods, positive_lambdas] / ridge_reference - 1)


def lasso_lower_bound(X, y, weights, lam):
    dual = (2 / len(y)) * (y - predict(X, weights))
    dual -= dual.mean()
    max_correlation = np.max(np.abs(X[:, 1:].T @ dual))
    if max_correlation > lam:
        dual *= lam / max_correlation
    return y @ dual - (len(y) / 4) * (dual @ dual)


lasso_bounds = {}
for lam in positive_lambdas:
    lower_bound = max(
        lasso_lower_bound(X_train, y_train, models[method, lam], lam)
        for method in lasso_methods
    )
    lasso_bounds[lam] = lower_bound

lasso_reference = pd.Series(lasso_bounds)
lasso_excess = 100 * (objectives.loc[lasso_methods, positive_lambdas] / lasso_reference - 1)
print("Training objective excess (%) over CF for Ridge and over a common lower bound for Lasso")
objective_table = pd.concat([ridge_excess.clip(lower=0), lasso_excess])
objective_table = objective_table.rename(columns=lambda lam: f"{lam:g}").rename_axis("Method")
print(objective_table.to_string(float_format="{:,.2f}".format))

Training objective excess (%) over CF for Ridge and over a common lower bound for Lasso
            0.01      0.1     1   10  100  1000
Method                                         
GD-R        0.00     0.00  0.00 0.00 0.00  8.44
SGD-R       4.66     3.65  3.02 3.98 3.58  1.19
GD-L   10,235.36 1,024.74 74.09 3.36 0.28  0.00
SGD-L  10,995.08 1,101.55 83.24 7.87 7.03  2.29


## 8. Lasso Sparsity Analysis

Table 2 counts exact nonzero coefficients out of the $p-1$ penalized terms. The intercept is excluded. A pair $(a,b)$ denotes the centered product $T_a(u)T_b(v)$. Terms with $b=0$ depend only on temperature, terms with $a=0$ depend only on humidity, and pairs with both indices positive represent interactions. The Boolean matrix `lasso_support` gives the status of every term for every method and lambda. For example, `lasso_support.index[lasso_support[("GD-L", 100)].to_numpy()].tolist()` returns all active pairs for GD-L at $\lambda=100$. Applying `~` to the Boolean mask gives the zero terms.

In [12]:
nonzero_counts = pd.DataFrame(index=lasso_methods, columns=lambdas, dtype=int)
for method in lasso_methods:
    for lam in lambdas:
        nonzero_counts.loc[method, lam] = np.count_nonzero(models[method, lam][1:])

print(f"Table 2. Nonzero penalized weights (out of {p - 1:,})")
sparsity_table = nonzero_counts.astype(int).rename(columns=lambda lam: f"{lam:g}").rename_axis("Method")
print(sparsity_table.to_string(formatters=["{:,}".format] * len(lambdas)))

Table 2. Nonzero penalized weights (out of 2,555)
           0  0.01   0.1     1    10   100  1000
Method                                          
GD-L   2,555 2,554 2,458 1,229   386    70     1
SGD-L  2,555 2,555 2,555 2,555 2,552 2,526 1,102


In [13]:
basis_index = pd.MultiIndex.from_tuples(orders[1:], names=["Temperature order", "Humidity order"])
lasso_support = pd.DataFrame(
    {(method, lam): models[method, lam][1:] != 0 for method in lasso_methods for lam in lambdas},
    index=basis_index
)
lasso_support.columns.names = ["Method", "Lambda"]

# Every coordinate has an exact active/zero entry in lasso_support.
for method in lasso_methods:
    for lam in [100, 1000]:
        weights = models[method, lam][1:]
        active = np.flatnonzero(lasso_support[method, lam])
        strongest = active[np.argsort(-np.abs(weights[active]))[:8]]
        label = "active terms" if len(active) <= len(strongest) else "largest active terms"
        print(f"{method}, lambda {lam:g}, {label}: {[orders[i + 1] for i in strongest]}")

selected_orders = [(1, 0), (2, 0), (3, 0), (0, 1), (0, 2), (1, 1), (2, 1)]
for method in lasso_methods:
    active_table = lasso_support[method].loc[selected_orders].copy()
    active_table.index = pd.Index([f"({a}, {b})" for a, b in selected_orders], name="Term")
    active_table.columns = [f"{lam:g}" for lam in active_table.columns]
    print()
    print(f"{method} status by lambda (True = active)")
    print(active_table.to_string())

GD-L, lambda 100, largest active terms: [(2, 2), (3, 0), (1, 0), (3, 2), (0, 4), (21, 37), (48, 22), (70, 0)]
GD-L, lambda 1000, active terms: [(3, 0)]
SGD-L, lambda 100, largest active terms: [(3, 0), (2, 2), (1, 0), (3, 2), (0, 4), (21, 37), (4, 2), (48, 22)]
SGD-L, lambda 1000, largest active terms: [(3, 0), (37, 0), (0, 45), (0, 19), (0, 66), (23, 13), (10, 0), (45, 0)]

GD-L status by lambda (True = active)
           0  0.01   0.1      1     10    100   1000
Term                                                
(1, 0)  True  True  True   True   True   True  False
(2, 0)  True  True  True  False  False  False  False
(3, 0)  True  True  True   True   True   True   True
(0, 1)  True  True  True   True  False  False  False
(0, 2)  True  True  True   True   True  False  False
(1, 1)  True  True  True  False  False  False  False
(2, 1)  True  True  True  False  False  False  False

SGD-L status by lambda (True = active)
           0  0.01   0.1     1    10   100   1000
Term             

## 9. Discussion and Conclusion

**Interpolation and overfitting.** Repeated temperatures put a floor of 716.95 rentals per day on temperature-only training RMSE. Adding humidity and using degree-70 Chebyshev products gave the design matrix full row rank (584). Unregularized least squares reached $3.58\times 10^{-8}$ training RMSE and 32,364,181.13 test RMSE. The training matrix remained ill-conditioned, with a condition number near $7.0\times 10^6$. The extreme test error shows that fitting all training days with a highly flexible polynomial can generalize poorly on held-out days.

**Effect of regularization.** Every method had a lower test RMSE at some positive `lambda` than at its own unregularized setting. The observed minimum for closed-form Ridge was 1,644.46 at `lambda = 1`. The lowest test RMSE overall was **1,235.90 rentals per day**, from SGD-L at `lambda = 100`. GD-L was close at 1,238.01. At `lambda = 1000`, test RMSE rose across all five methods relative to their respective observed minima, and training RMSE also rose. Ridge shrank the coefficients with a quadratic penalty. Lasso could additionally set coefficients exactly to zero. These minima describe the fixed test split.

**Optimization.** All objectives and predictions stayed finite. At zero penalty, the iterative fits still had training RMSE between 602 and 625 at the common budget, so their test errors describe partially optimized fits. Small singular values make some training directions slow to fit. At positive penalties through 100, the GD-R objective was within 0.01% of CF. At 1000 the gap was 8.44%, since the shared step size was small and the intercept approached the training mean slowly. SGD-R was within about 4.7% of the CF objective for every positive penalty. SGD-L's training objective was about 2% to 8% above GD-L's across the positive grid. The Lasso lower bounds were loose at small penalties, so the similar objectives there give limited evidence of proximity to the minimum.

**Sparsity and interpretation.** At `lambda = 100`, GD-L kept 70 of 2,555 penalized terms and SGD-L kept 2,526. GD-L retained temperature terms $(1,0)$ and $(3,0)$, a humidity term $(0,4)$, and interactions such as $(2,2)$ and $(3,2)$. These five terms also had the largest coefficient magnitudes in SGD-L at the same penalty. GD-L set $(2,0)$, $(0,1)$, and $(1,1)$ to zero, while SGD-L retained them. The temperature terms describe a nonlinear response, and the interactions allow the fitted temperature response to vary with humidity. These terms can also reflect shared seasonal patterns and are descriptive associations. At `lambda = 1000`, GD-L kept only $(3,0)$, while SGD-L still kept 1,102 terms, including the pure humidity term $(0,45)$. Even with a decreasing step size, an individual sample gradient can reactivate a coefficient whose full-batch update would keep it at zero.

## 10. Execution Instructions

1. From the directory containing this notebook and the data folder, install the dependencies.

   ```bash
   python -m pip install numpy pandas notebook
   ```

2. Start Jupyter from that directory. A single BLAS thread reduces overhead on repeated matrix-vector operations when using OpenBLAS.

   ```bash
   OPENBLAS_NUM_THREADS=1 jupyter notebook
   ```

3. Open **linear_regression_part2.ipynb**, choose **Kernel → Restart Kernel and Run All Cells**, and save the notebook to retain the results. The training cell may take a few minutes. Once the dependencies are installed, execution uses local **data/day.csv** and runs offline.

## 11. References

- Fanaee-T, H. (2013). *Bike Sharing* [Dataset]. UCI Machine Learning Repository. [Dataset documentation](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)
- Fanaee-T, H., & Gama, J. *Event labeling combining ensemble detectors and background knowledge*. Progress in Artificial Intelligence. [Original study](https://doi.org/10.1007/s13748-013-0040-3)
- UCI Machine Learning Repository. *Bike Sharing dataset archive and Readme.txt*. [Original archive](https://archive.ics.uci.edu/static/public/275/bike%2Bsharing%2Bdataset.zip)